# RSCT-MAST Quickstart

Compare MAST binary failure annotations with GeoCert structured certificates.

**Requirements**: `numpy` (that's it)

## 1. Load MAST Traces

61 human-annotated multi-agent traces (31 AG2, 30 HyperAgent) from [Cemri et al. 2025](https://arxiv.org/abs/2503.13657).

In [ ]:
import sys
sys.path.insert(0, '..')

from load_mast import load_all_traces, MAST_CATEGORIES, MODE_TO_CATEGORY

traces = load_all_traces()
print(f"Loaded {len(traces)} traces")
print(f"  AG2: {sum(1 for t in traces if t['source'] == 'AG2')}")
print(f"  HyperAgent: {sum(1 for t in traces if t['source'] == 'HyperAgent')}")

## 2. Extract Structural Signals

17 scalar signals computed directly from trajectory structure. No labels, no classification — just measurement.

In [ ]:
from signals import extract_signals
import numpy as np

for t in traces:
    t['signals'] = extract_signals(t)

# Show signals for first trace
print(f"Trace: {traces[0]['instance_id']} ({traces[0]['source']})")
print(f"Steps: {traces[0]['n_steps']}")
print(f"Failures: {sum(traces[0]['annotations'].values())}")
print("\nSignals:")
for k, v in sorted(traces[0]['signals'].items()):
    print(f"  {k:25s} {v:.3f}")

## 3. Category Separation

Are MAST's three failure categories separable in signal space?

In [ ]:
from collections import defaultdict

# Group traces by MAST category
cat_traces = defaultdict(list)
baseline = []

for t in traces:
    active = [MODE_TO_CATEGORY.get(m, 'unknown') for m, v in t['annotations'].items() if v]
    cats = set(active)
    if not cats:
        baseline.append(t)
    for c in cats:
        cat_traces[c].append(t)

# Signal keys (exclude n_steps which has different scale)
sig_keys = sorted(k for k in traces[0]['signals'].keys() if k != 'n_steps')

def centroid(trace_list):
    mat = np.array([[t['signals'][k] for k in sig_keys] for t in trace_list])
    return mat.mean(axis=0), mat.std(axis=0) + 1e-8

# Compute centroids
groups = {}
for name, tlist in cat_traces.items():
    groups[name] = centroid(tlist)
if baseline:
    groups['baseline'] = centroid(baseline)

# Pairwise z-score distances
names = sorted(groups.keys())
print("Pairwise centroid distances (z-score space):")
print()
for i, a in enumerate(names):
    for b in names[i+1:]:
        mu_a, std_a = groups[a]
        mu_b, std_b = groups[b]
        pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)
        dist = np.sqrt(np.sum(((mu_a - mu_b) / pooled_std)**2))
        print(f"  {a:30s} vs {b:30s} : {dist:.3f}")

## 4. GeoCert Failure Taxonomy

9 fine-grained evaluation-failure modes in 3 categories. Each mode has expected signal patterns and diagnostic tests.

In [ ]:
from stress_geocert import get_geocert_taxonomy

taxonomy = get_geocert_taxonomy()
for cat_id, cat in taxonomy['categories'].items():
    print(f"\n{cat_id} — {cat['name']}")
    for mode in cat['modes']:
        print(f"  {mode['id']}: {mode['name']}")
        print(f"         {mode['description'][:80]}...")

## 5. Injection-Detection Stress Suite

For each of the 9 failure modes: inject synthetic signals matching the mode, then diagnose. Perfect separation = 100% top-1 accuracy.

In [ ]:
from stress_geocert import run_geocert_stress_suite

results = run_geocert_stress_suite(intensity=0.85, seed=3500)

print(f"Stress Suite Results:")
print(f"  Modes tested: {results['total_modes']}")
print(f"  Top-1 accuracy: {results['top1_accuracy']:.0%}")
print(f"  Top-3 accuracy: {results['top3_accuracy']:.0%}")
print()

for r in results['results']:
    match = 'PASS' if r['top1_match'] else 'FAIL'
    conf = r['candidates'][0]['confidence']
    print(f"  [{match}] {r['injected_mode']:8s} {r['injected_name']:30s} -> {r['top1_mode']:8s} (conf={conf:.3f})")

## 6. Diagnose a Custom Signal Record

Feed arbitrary signal values and get a typed failure diagnosis with confidence scores.

In [ ]:
from stress_geocert import diagnose_geocert_failure

# Example: signals suggesting scalar projection failure
custom_signals = {
    'alpha_rank_gap': 0.82,
    'sigma_outlier': 0.75,
    'kappa_compression': 0.40,
    'gate_entropy': 0.15,
    'aggregate_separation': 0.60,
}

diagnosis = diagnose_geocert_failure(custom_signals, top_k=5)

print(f"Top diagnosis: {diagnosis['top1_mode']}")
print()
for c in diagnosis['candidates']:
    print(f"  {c['mode']:8s} {c['name']:30s} conf={c['confidence']:.3f}  ({c['category']})")

## 7. The Dimensionality Argument

MAST uses 22 binary annotations per trace. GeoCert produces continuous n-attribute certificates per (solver x target) pair. The certificate space is fundamentally higher-dimensional than binary labeling — structured certificates can discover failure modes that annotation schemes cannot represent.

In [ ]:
# Count unique annotation patterns vs continuous signal diversity
patterns = set()
for t in traces:
    pattern = tuple(sorted(m for m, v in t['annotations'].items() if v))
    patterns.add(pattern)

sig_mat = np.array([[t['signals'][k] for k in sig_keys] for t in traces])
sig_rank = np.linalg.matrix_rank(sig_mat, tol=0.01)

print(f"MAST: {len(patterns)} unique annotation patterns across {len(traces)} traces")
print(f"Signals: {len(sig_keys)} features, effective rank = {sig_rank}")
print(f"\nBinary labels collapse a continuous space into {len(patterns)} bins.")
print(f"Certificate signals preserve {sig_rank} independent axes of variation.")